# 00 · Configuración del entorno

Funciona en **Google Colab** y en **escritorio** (VS Code / Jupyter).

1. Edita **solo** la celda de configuración.
2. En Colab, guarda tu token de GitHub en *Secrets* (icono de llave) con el nombre `GITHUB_TOKEN` y tu clave de W&B como `WANDB_API_KEY`.
3. Ejecuta todo (*Runtime → Run all*).

## 1. Configuración (editar)

In [ ]:
GITHUB_USER = "TU_USUARIO"          # dueño del repo en GitHub
GITHUB_REPO = "ProyectoIA_ComandosDeVoz"
GIT_NAME    = "Tu Nombre"
GIT_EMAIL   = "tu_correo@ejemplo.com"

DRIVE_FOLDER = "ProyectoIA_ComandosDeVoz"   # carpeta compartida en Drive
# Ruta local de Google Drive para escritorio (solo si no usas Colab), p. ej.:
#   Windows: r"G:\Mi unidad"   macOS: "/Users/tu/Library/CloudStorage/GoogleDrive-.../Mi unidad"
LOCAL_DRIVE_ROOT = r"G:\Mi unidad"

WANDB_PROJECT = "comandos-de-voz"
WANDB_ENTITY  = None   # nombre del equipo en W&B, o None

## 2. Detectar entorno y montar Drive

In [ ]:
import os, sys, subprocess
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_ROOT = Path("/content/drive/MyDrive")
else:
    DRIVE_ROOT = Path(LOCAL_DRIVE_ROOT)

PROJECT_DRIVE = DRIVE_ROOT / DRIVE_FOLDER
DATA_DIR      = PROJECT_DRIVE / "speech_commands_v0.02"
CKPT_DIR      = PROJECT_DRIVE / "checkpoints"
EXPORT_DIR    = PROJECT_DRIVE / "exports"
for d in (CKPT_DIR, EXPORT_DIR):
    d.mkdir(parents=True, exist_ok=True)

print("Colab:", IN_COLAB)
print("Carpeta del proyecto en Drive:", PROJECT_DRIVE, "| existe:", PROJECT_DRIVE.exists())

## 3. Clonar o actualizar el repositorio

In [ ]:
def _secret(name):
    if IN_COLAB:
        from google.colab import userdata
        try:
            return userdata.get(name)
        except Exception:
            return None
    return os.environ.get(name)

def run(cmd, cwd=None):
    r = subprocess.run(cmd, cwd=cwd, shell=True, capture_output=True, text=True)
    out = (r.stdout + r.stderr).strip()
    token = _secret("GITHUB_TOKEN")
    if token:
        out = out.replace(token, "***")
    print(out)
    return r.returncode

if IN_COLAB:
    REPO_DIR = Path("/content") / GITHUB_REPO
    token = _secret("GITHUB_TOKEN")
    auth = f"{token}@" if token else ""
    remote = f"https://{auth}github.com/{GITHUB_USER}/{GITHUB_REPO}.git"
    if not REPO_DIR.exists():
        run(f"git clone {remote} {REPO_DIR}")
    else:
        run("git pull --ff-only", cwd=REPO_DIR)
    run(f'git config user.name "{GIT_NAME}"', cwd=REPO_DIR)
    run(f'git config user.email "{GIT_EMAIL}"', cwd=REPO_DIR)
else:
    # En escritorio se asume que el notebook se abre desde el repo clonado (notebooks/).
    REPO_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
print("Repo:", REPO_DIR)

## 4. Instalar librerías

In [ ]:
if IN_COLAB:
    run("pip install -q -r requirements.txt")
else:
    print("En escritorio: instala PyTorch desde pytorch.org y luego `pip install -r requirements.txt` en tu entorno.")

## 5. Semillas y dispositivo

In [ ]:
import torch
from src.config import COMMANDS, SEED, set_seed

set_seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("torch", torch.__version__, "| dispositivo:", DEVICE)

## 6. Verificar el dataset

In [ ]:
problemas = []
if not DATA_DIR.exists():
    problemas.append(f"No existe {DATA_DIR}")
else:
    for c in COMMANDS:
        n = len(list((DATA_DIR / c).glob("*.wav"))) if (DATA_DIR / c).exists() else 0
        print(f"{c:>6}: {n} archivos")
        if n == 0:
            problemas.append(f"Falta la carpeta o está vacía: {c}")
    for lst in ("validation_list.txt", "testing_list.txt"):
        p = DATA_DIR / lst
        if p.exists():
            print(f"{lst}: {sum(1 for _ in open(p))} líneas")
        else:
            problemas.append(f"Falta {lst}")

print("\nOK" if not problemas else "\nPROBLEMAS:\n- " + "\n- ".join(problemas))

## 7. Iniciar sesión en Weights & Biases

In [ ]:
import wandb
key = _secret("WANDB_API_KEY")
wandb.login(key=key) if key else wandb.login()

## 8. Ayudante para subir cambios desde Colab

In [ ]:
def git_push(mensaje, rama=None):
    """Hace add + commit + push. Úsalo en tu rama, nunca directo a main sin revisión."""
    if rama:
        run(f"git checkout -B {rama}")
    run("git add -A")
    run(f'git commit -m "{mensaje}"')
    run("git pull --rebase")
    run(f"git push -u origin HEAD")

# Ejemplo:
# git_push("Fase 1: preprocesamiento", rama="fase1-preprocesamiento")